# 1. Charger les données

In [10]:
import pandas as pd
import numpy as np

from pathlib import Path

# Chemin du projet
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

# Chargement des données
train = pd.read_parquet(root / "data/processed/train.parquet")
val = pd.read_parquet(root / "data/processed/validation.parquet")
test = pd.read_parquet(root / "data/processed/test.parquet")

print("Train :", train.shape)
print("Validation :", val.shape)
print("Test :", test.shape)

Train : (199364, 31)
Validation : (42721, 31)
Test : (42722, 31)


# 2. Séparer les variables X et la cible

In [11]:
X_train = train.drop(columns="Class")
y_train = train["Class"]

X_val = val.drop(columns="Class")
y_val = val["Class"]

X_test = test.drop(columns="Class")
y_test = test["Class"]

print(X_train.shape)
print(y_train.shape)

(199364, 30)
(199364,)


# 3. Créer le pipeline de preprocessing
Nous allons utiliser trois composants de scikit-learn :
- SimpleImputer : remplace les éventuelles valeurs manquantes.
- StandardScaler : centre et réduit les variables.
- ColumnTransformer : applique des traitements différents selon les colonnes.

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Variables à standardiser
numeric_features = ["Time", "Amount"]

# Variables PCA
pca_features = [f"V{i}" for i in range(1, 29)]

# Pipeline Time et Amount
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline des variables PCA
pca_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Nouveau preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("pca", pca_pipeline, pca_features)
    ],
    verbose_feature_names_out=False
)

# 4. Appliquer les transformations

In [13]:
# Apprendre les paramètres uniquement sur le train
X_train_processed = preprocessor.fit_transform(X_train)

# Appliquer les mêmes paramètres aux autres jeux
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train :", X_train_processed.shape)
print("Validation :", X_val_processed.shape)
print("Test :", X_test_processed.shape)

Train : (199364, 30)
Validation : (42721, 30)
Test : (42722, 30)


### Point essentiel en MLOps : on ne fait jamais fit_transform() séparément sur les données de test, car cela entraînerait une fuite d'informations (data leakage).

# 5. Vérifier le résultat

In [14]:
columns = preprocessor.get_feature_names_out()

df_processed = pd.DataFrame(
    X_train_processed,
    columns=columns
)

print(df_processed.head())

print("\nMoyennes :")
print(df_processed[["Time", "Amount"]].mean())

print("\nÉcarts-types :")
print(df_processed[["Time", "Amount"]].std(ddof=0))

       Time    Amount        V1        V2        V3        V4        V5  \
0 -2.041695  0.240443 -1.359807 -0.072781  2.536347  1.378155 -0.169567   
1 -2.041695 -0.349849  1.191857  0.266151  0.166480  0.448154  0.060018   
2 -2.041666  1.160611 -1.358354 -1.340163  1.773209  0.379780 -0.503198   
3 -2.041666  0.135505 -0.966272 -0.185226  1.792993 -0.863291 -0.010309   
4 -2.041637 -0.079471 -1.158233  0.877737  1.548718  0.403034 -0.407193   

         V6        V7        V8  ...       V19       V20       V21       V22  \
0  0.462388  0.239599  0.098698  ...  0.403993  0.251412 -0.018307  0.277838   
1 -0.082361 -0.078803  0.085102  ... -0.145783 -0.069083 -0.225775 -0.638672   
2  1.800499  0.791461  0.247676  ... -2.261857  0.524980  0.247998  0.771679   
3  1.247203  0.237609  0.377436  ... -1.232622 -0.208038 -0.108300  0.005274   
4  0.095921  0.592941 -0.270533  ...  0.803487  0.408542 -0.009431  0.798278   

        V23       V24       V25       V26       V27       V28  
0 -0

### Les moyennes de Time et Amount doivent être proches de zéro et leurs écarts-types proches de 1.

# 6. Sauvegarder le pipeline

In [15]:
import joblib

model_dir = root / "models"
model_dir.mkdir(exist_ok=True)

joblib.dump(
    preprocessor,
    model_dir / "preprocessor.joblib"
)

print("Pipeline sauvegardé.")

Pipeline sauvegardé.


# 7. Vérifier que le pipeline fonctionne après rechargement

In [16]:
loaded_preprocessor = joblib.load(
    model_dir / "preprocessor.joblib"
)

# Transformer cinq nouvelles observations
sample = X_test.iloc[:5]

result = loaded_preprocessor.transform(sample)

print("Dimensions :", result.shape)
print(result)

Dimensions : (5, 30)
[[ 2.34425454 -0.19212167  2.28872687 -1.29619303 -1.6174443  -1.59217041
  -0.75022796 -0.81201234 -0.6525446  -0.29087127 -1.45550557  1.6025636
  -1.14628624 -1.1960339  -0.34623668 -0.01937603 -0.31637163 -0.88083258
   0.69468927 -0.43860762  0.04773562 -0.42147178 -0.30461731 -0.49350792
   0.25540221  0.556435   -0.0851113  -0.26102771 -0.02961491 -0.0516267 ]
 [ 2.34428353 -0.35346447  2.01917898  0.14371029 -1.59488598  0.34153096
   0.42017443 -0.65340398  0.11566219 -0.12577642  0.1567027  -0.22216085
   1.37135186  1.12459561  0.37458462 -0.72169969 -0.55586695  0.43814079
   0.26639575  0.07409165  0.26890003 -0.12348523 -0.29567095 -0.76267289
   0.34536265  0.68151066 -0.29875745  0.14113967 -0.06347861 -0.03780633]
 [ 2.34437047 -0.35346447  2.00446626 -0.12048563 -1.01391818  0.31455149
  -0.11152643 -0.85673578  0.05674833 -0.15723609  0.30173089  0.23882024
   1.05279778  1.07954257 -0.21891887  0.51847645 -0.5995244   0.01647312
  -0.44152595 -0